# NHAMCS Scenario 1 — Waiting, staying, boarding: the flow of patients through the ED

_Describing ED operations with a national sample. Level: first analysis on this dataset. About 90 minutes._

**Before you start.** NHAMCS is a national probability sample of US emergency department visits:
86,864 visits abstracted from the records of about 200 hospitals a year, 2018 to 2022. Every record
carries a weight, the number of US visits it stands for, and **the MIP does not use weights**: every
figure in this notebook describes *the sampled visits*, not US emergency care. CDC codes are kept:
questions are 1 = Yes, 2 = No; checkbox items 0 = not marked, 1 = marked; Sex is 1 = Female,
2 = Male; unknown is 98. One row is one visit, and the same person may appear twice.

**How this notebook works.** Every analysis is a *federated* call: the code selects variables,
defines a cohort with filters and asks the MIP for aggregates (counts, means, a test, a model). No
visit record ever reaches this notebook. Run the cells in order; each step says what to look for
and what result to expect.

**Learning goals.** Describe skewed time variables with medians and quartiles rather than means;
compare a mean across five groups with one-way ANOVA; relate triage level to waiting time,
length of visit and admission; test a two-group difference with a t-test; and explain boarding as
an ED quality problem.

**Clinical background.** Emergency departments sort arriving patients by urgency at **triage**:
level 1 (immediate) to 5 (non-urgent). The higher the acuity, the shorter the wait to a physician
should be. The **length of visit** depends on the work-up and on the disposition: admitted patients
stay longest, and when no inpatient bed is free they wait in the ED after the admission decision,
a phenomenon called **boarding**, which crowds the ED and worsens outcomes. NHAMCS records waiting
time to the first provider, total length of visit, and for admitted patients the boarding time,
all in minutes. Waiting time is missing for 14% of visits; triage level is missing or unknown for
a quarter of visits and some emergency service areas do not triage (code 7).

## 1. Connect to the MIP and open the NHAMCS pathology

In [ ]:
import mip
from mip.filters import F

client = mip.Client.from_env()          # connection configured by the MIP portal
catalog = client.catalog()

try:
    dm = catalog.data_model("NHAMCS_ED")   # add version="v1" if the catalogue holds several versions
except LookupError:
    display(catalog.summaries())           # the pathology label differs: pick it from this list
    raise
ed = dm.datasets.list()[0]              # one dataset: the five survey years together
display(dm)
display(ed)

In [ ]:
# The variable tree follows the visit: Survey administration, Arrival and visit timing, Patient,
# Expected source of payment, Triage and initial vital signs, Reason for visit, Injury, Diagnoses,
# Chronic conditions, Diagnostic services, Procedures, Medications, Vital signs after triage,
# Providers seen, Disposition, ED and hospital characteristics.
dm.tree()

## 2. Toolbox

In [ ]:
# ---- Toolbox: small helpers that turn MIP results into readable tables and charts. ----
# Run this cell once. You do not need to read it to follow the scenario; come back to it
# when you want to see how a result payload is unpacked.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from mip.preprocessing import MissingValuesHandler

POOLED = "all datasets"                 # the row describe() adds across the datasets of the pathology
PCT, MIN = "{:.1f}", "{:.0f}"           # number formats of the tables: percentages, minutes


def analysis(variables, filters=None, drop_missing=False):
    """A Pipeline on the NHAMCS dataset with the given variables and cohort filter.
    Tests and models need drop_missing=True: the platform requires an explicit missing-values
    step and analyses complete cases only. describe() keeps it off so it can count missing values."""
    handler = MissingValuesHandler(strategies={v: "drop" for v in variables}) if drop_missing else None
    return mip.Pipeline(analysis_set=dm.select(datasets=[ed], variables=variables), filters=filters,
                        handle_missing=handler)


def labels_of(variable):
    """{code: label} of a categorical variable, whatever format the catalogue uses."""
    raw = variable._data.get("enumerations") or {}
    if isinstance(raw, dict):
        return {str(k): str(v) for k, v in raw.items()}
    return {str(item.get("code") if isinstance(item, dict) else item):
            str(item.get("label") if isinstance(item, dict) else item) for item in raw}


def rows_for(result, variable):
    """The pooled 'describe' row of one variable (the platform adds one row per dataset as well)."""
    return [r for r in result.raw.get("featurewise", [])
            if r.get("variable") == variable._code and r.get("dataset") == POOLED]


def numeric_summary(result, variables):
    """n, missing, mean, SD, quartiles and range of numerical variables from a describe result."""
    rows = []
    for v in variables:
        for r in rows_for(result, v):
            d = r.get("data") or {}
            rows.append({"variable": v.label, "n": d.get("num_dtps"), "missing": d.get("num_na"),
                         "mean": d.get("mean"), "sd": d.get("std"), "q1": d.get("q1"), "median": d.get("q2"),
                         "q3": d.get("q3"), "min": d.get("min"), "max": d.get("max")})
    return pd.DataFrame(rows)


def category_counts(result, variable):
    """Counts and percentages of the levels of a categorical variable from a describe result."""
    lab = labels_of(variable)
    rows = []
    for r in rows_for(result, variable):
        counts = (r.get("data") or {}).get("counts") or {}
        total = sum(counts.values())
        for c, n in counts.items():
            rows.append({"variable": variable.label, "level": lab.get(str(c), str(c)), "count": n,
                         "percent": 100 * n / total if total else np.nan})
    return pd.DataFrame(rows)


def yes_percent(result, variable, level="Yes"):
    """Percentage of one level (default Yes) of a categorical variable in a describe result."""
    k = category_counts(result, variable)
    return float(k.loc[k["level"] == level, "percent"].sum()) if len(k) else np.nan


def by_level(numeric, group, levels=None, base_filter=None):
    """n, mean, median and quartiles of a numerical variable in each level of a categorical one
    (one describe per level). levels: the codes to use, by default every code of the variable."""
    lab = labels_of(group)
    rows = []
    for c in levels or lab:
        f = F(group) == c
        if base_filter is not None:
            f = f & base_filter
        res = analysis([numeric], f).describe()
        for r in rows_for(res, numeric):
            d = r.get("data") or {}
            rows.append({group.label: lab.get(c, c), "n": d.get("num_dtps"), "mean": d.get("mean"),
                         "q1": d.get("q1"), "median": d.get("q2"), "q3": d.get("q3")})
    return pd.DataFrame(rows)


def crosstab(exposure, outcome, base_filter=None, event="1", levels=None):
    """Cross-tabulation of outcome by exposure, as the MIP does it: counts come from one
    describe per exposure level, the p-value from the Chi-squared Test on the whole cohort."""
    lab_x, lab_y = labels_of(exposure), labels_of(outcome)
    rows = []
    for c in levels or lab_x:
        f = (F(exposure) == c) & F(outcome).is_not_null()
        if base_filter is not None:
            f = f & base_filter
        res = analysis([exposure, outcome], f).describe()
        counts = {}
        for r in rows_for(res, outcome):
            counts = (r.get("data") or {}).get("counts") or {}
        n = sum(counts.values())
        if n == 0:
            continue
        row = {exposure.label: lab_x.get(c, c), "n": n}
        for yc, ylabel in lab_y.items():
            row[ylabel] = counts.get(yc, 0)
        row["% " + lab_y.get(event, event)] = 100 * counts.get(event, 0) / n
        rows.append(row)
    table = pd.DataFrame(rows)
    f = F(exposure).is_not_null() & F(outcome).is_not_null()
    if levels:
        f = f & F(exposure).isin(list(levels))
    if base_filter is not None:
        f = f & base_filter
    chi = analysis([exposure, outcome], f, drop_missing=True).chi_square_test(x=exposure, y=outcome)
    stats = chi.raw if isinstance(chi.raw, dict) else {}
    table.attrs["chi2"] = stats.get("chi2")
    table.attrs["p"] = stats.get("p_value", stats.get("p"))
    table.attrs["outcome"] = outcome.label
    return table


def rates_by(group, outcomes, levels=None, base_filter=None, event="1"):
    """% of visits with each outcome (code `event`) in every level of `group`: one describe per
    level for the counts of all outcomes, one Chi-squared Test per outcome (in .attrs)."""
    lab = labels_of(group)
    rows = []
    for c in levels or lab:
        f = F(group) == c
        if base_filter is not None:
            f = f & base_filter
        res = analysis(outcomes, f).describe()
        row = {group.label: lab.get(c, c)}
        for y in outcomes:
            counts = {}
            for r in rows_for(res, y):
                counts = (r.get("data") or {}).get("counts") or {}
            n = sum(counts.values())
            row.setdefault("n", n)
            row[y.label] = 100 * counts.get(event, 0) / n if n else np.nan
        rows.append(row)
    table = pd.DataFrame(rows)
    table.attrs["chi2"], table.attrs["p"] = {}, {}
    for y in outcomes:
        f = F(group).is_not_null() & F(y).is_not_null()
        if levels:
            f = f & F(group).isin(list(levels))
        if base_filter is not None:
            f = f & base_filter
        chi = analysis([group, y], f, drop_missing=True).chi_square_test(x=group, y=y)
        table.attrs["chi2"][y.label] = chi.raw.get("chi2")
        table.attrs["p"][y.label] = chi.raw.get("p_value")
    return table


def band_rates(outcomes, bands, base_filter=None, event="1"):
    """% of visits with each outcome in bands of a variable: bands maps a label to a filter."""
    rows = []
    for label, f in bands.items():
        if base_filter is not None:
            f = f & base_filter
        res = analysis(outcomes, f).describe()
        row = {"band": label}
        for y in outcomes:
            counts = {}
            for r in rows_for(res, y):
                counts = (r.get("data") or {}).get("counts") or {}
            n = sum(counts.values())
            row.setdefault("n", n)
            row[y.label] = 100 * counts.get(event, 0) / n if n else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "n/a"
    return "< 0.001" if p < 0.001 else f"{p:.3f}"


def show_crosstab(table, title=None):
    """Display a crosstab table, its chi-square test and a bar chart of the percentages."""
    pct_col = [c for c in table.columns if c.startswith("% ")][0]
    display(table.style.format({pct_col: "{:.1f}"}).hide(axis="index"))
    print(f"Chi-squared Test: chi2 = {table.attrs['chi2']:.1f}, p = {fmt_p(table.attrs['p'])}")
    ax = table.set_index(table.columns[0])[pct_col].plot(kind="bar", color="#4C72B0", figsize=(6, 3))
    ax.set_ylabel(pct_col)
    ax.set_title(title or f"{table.attrs['outcome']} by {table.columns[0]}")
    ax.set_xlabel("")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


def anova_summary(result):
    """One-way ANOVA result -> F, p, n, a table of the group means and SDs, and the Tukey comparisons."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    table, ci = raw.get("anova_table", {}), raw.get("ci_info", {})
    groups = pd.DataFrame({"mean": pd.Series(ci.get("means", {})), "sd": pd.Series(ci.get("sample_stds", {}))})
    return table.get("f_stat"), table.get("p_value"), table.get("n_obs"), groups, pd.DataFrame(raw.get("tuckey_test", []))


def ttest_summary(result):
    """Independent t-test result -> t, p, df, mean difference with its 95% CI, Cohen's d, n."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    return {k: raw.get(k) for k in ("t_stat", "p", "df", "mean_diff", "ci_lower", "ci_upper", "cohens_d", "n_obs")}


def odds_ratio_table(result, variables):
    """Logistic regression result -> one row per term with odds ratio, 95% CI and p.
    Categorical covariates appear as 'variable = level' relative to the reference level
    (the first code of the variable) that the platform leaves out."""
    by_code = {v._code: v for v in variables}

    def pretty(term):
        term = str(term)
        if "[" in term and term.endswith("]"):
            base, level = term[:-1].split("[", 1)
            v = by_code.get(base)
            return f"{v.label} = {labels_of(v).get(level, level)}" if v else term
        v = by_code.get(term)
        return f"{v.label} (per unit)" if v else term

    frame = result.to_frame().copy()
    frame = frame[~frame["feature"].astype(str).str.lower().isin(["intercept", "const"])]
    frame["term"] = frame["feature"].map(pretty)
    for col in ("odds_ratio", "or_lower_ci", "or_upper_ci", "p"):
        if col not in frame.columns:
            frame[col] = np.nan
    out = frame[["term", "odds_ratio", "or_lower_ci", "or_upper_ci", "p"]]
    return out.rename(columns={"or_lower_ci": "ci_low", "or_upper_ci": "ci_high"}).reset_index(drop=True)


def show_odds_ratios(table, title):
    """Table and forest plot (log scale) of odds ratios."""
    display(table.style.format({"odds_ratio": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": "{:.4f}"}).hide(axis="index"))
    t = table.iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(t) + 1.5))
    ax.errorbar(t["odds_ratio"], range(len(t)),
                xerr=[t["odds_ratio"] - t["ci_low"], t["ci_high"] - t["odds_ratio"]], fmt="o", capsize=4)
    ax.axvline(1, linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(t["term"])
    ax.set_xlabel("odds ratio (log scale)")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


print("Toolbox ready.")

## 3. The variables of this scenario

| Code | Label in the MIP | Tree location (under NHAMCS_ED) | Role | Values |
|---|---|---|---|---|
| `WAITTIME` | Wait time to provider | Arrival and visit timing | outcome: minutes from arrival to the first provider | integer, 0-1500 minutes |
| `LOV` | Length of visit | Arrival and visit timing | outcome: minutes from arrival to departure | integer, 0-6000 minutes |
| `BOARDED` | Boarding time | Arrival and visit timing | outcome: minutes after the admission decision; admitted visits only | integer, 0-5000 minutes |
| `IMMEDR` | Triage level | Triage and initial vital signs | grouping; cohort filter 1 to 5 | 0 No triage reported, ESA conducts nursing triage; 1 Immediate; 2 Emergent; 3 Urgent; 4 Semi-urgent; 5 Nonurgent; 7 Visit in an emergency service area that does not conduct nursing triage; 98 Unknown |
| `admitted` | Admitted to hospital | Disposition | outcome of the cross-tabulation; grouping; curation-derived | 0 No; 1 Yes |
| `ARREMS` | Arrival by ambulance | Arrival and visit timing | descriptive by triage level | 1 Yes; 2 No; 98 Unknown |
| `arr_period` | Arrival period | Arrival and visit timing | grouping; curation-derived | 1 Night, 00:00-07:59; 2 Day, 08:00-15:59; 3 Evening, 16:00-23:59 |
| `weekend` | Weekend visit | Survey administration | grouping; curation-derived | 0 No; 1 Yes |
| `YEAR` | Survey year | Survey administration | grouping (trends) | 2018 2018; 2019 2019; 2020 2020; 2021 2021; 2022 2022 |
| `LWBS` | Left without being seen | Disposition / Disposition checkboxes | outcome of the trend cross-tabulation | 0 No; 1 Yes |
| `PAYTYPER` | Primary expected payer | Expected source of payment | grouping (extension) | 1 Private insurance; 2 Medicare; 3 Medicaid, CHIP or other state-based program; 4 Worker's compensation; 5 Self-pay; 6 No charge/charity; 7 Other; 98 Unknown |
| `BOARD` | Admitted patients ever boarded more than 2 hours in the ED | ED and hospital characteristics | hospital-level variable (discussion) | 1 Yes; 2 No; 98 Unknown; 96 Refused to answer |
| `OBSCLIN` | ED has an observation or clinical decision unit | ED and hospital characteristics | hospital-level variable (discussion) | 1 Yes; 2 No; 98 Unknown; 96 Refused to answer |

Variables are picked by their **label**, as the tree shows it; the tree location tells you where
to find them in `dm.tree()` and in the MIP interface. The code (for example `WAITTIME`) is what the
csv column is called; you will see it in some result payloads.

In [ ]:
wait      = dm.variables["Wait time to provider"]
lov       = dm.variables["Length of visit"]
boarded   = dm.variables["Boarding time"]
triage    = dm.variables["Triage level"]
period    = dm.variables["Arrival period"]
weekend   = dm.variables["Weekend visit"]
admitted  = dm.variables["Admitted to hospital"]
ambulance = dm.variables["Arrival by ambulance"]
year      = dm.variables["Survey year"]
lwbs      = dm.variables["Left without being seen"]
display(mip.to_frame([wait, lov, boarded, triage, period, weekend, admitted, ambulance, year, lwbs]))

## Step 1. Describe the two time variables

One `describe()` on all visits gives, for each numerical variable, the number of recorded values,
the mean, the standard deviation and the quartiles. The histogram that follows shows why the
median, not the mean, is the right summary of a length of visit.

*Expected:* waiting time, 73,717 visits with a value: mean 35 minutes but median 14, quartiles 5 to
36. Length of visit, 81,984 values: mean 267 minutes, median 175, quartiles 102 to 290. The means
are more than double the medians: a few very long waits and stays pull them up. For such variables
report the median and quartiles; use the mean only when the test needs it.

In [ ]:
overview = analysis([wait, lov]).describe()
summary = numeric_summary(overview, [wait, lov])
display(summary.style.format({c: MIN for c in ("mean", "sd", "q1", "median", "q3", "min", "max")}).hide(axis="index"))

# The long tail: a histogram of the length of visit (60 bins over the whole range; bars under 10 visits are hidden)
h = analysis([lov], drop_missing=True).histogram(variable=lov, bins=60).raw["histogram"][0]
edges, counts = h["bins"], [c or 0 for c in h["counts"]]
row = summary.set_index("variable").loc["Length of visit"]
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.bar(edges[:-1], counts, width=np.diff(edges), align="edge", color="#4C72B0")
ax.axvline(row["median"], color="k", linestyle="--")
ax.axvline(row["mean"], color="#C44E52", linestyle="--")
ax.text(row["median"] + 20, max(counts) * 0.9, f"median {row['median']:.0f}", fontsize=9)
ax.text(row["mean"] + 20, max(counts) * 0.75, f"mean {row['mean']:.0f}", fontsize=9, color="#C44E52")
ax.set_xlim(0, 1500)
ax.set_xlabel("length of visit (minutes)"); ax.set_ylabel("visits"); ax.set_title("Length of visit: a long right tail")
plt.tight_layout()
plt.show()

## Step 2. Waiting time by triage level

Among triaged visits (levels 1 to 5, 58,124 visits): one `describe()` per level for the medians,
then *One-way ANOVA* of the waiting time across the five levels. One-way ANOVA extends the t-test
to more than two groups: its F statistic asks whether the group means differ more than chance
alone would make them differ.

*Expected medians:* level 1: 9 minutes; 2: 12; 3: 14; 4: 15; 5: 17. Means 27, 29, 36, 32, 35.
ANOVA F about 20, p < 0.001, on 51,424 visits with a recorded wait. The gradient is in the
expected direction and real, but small: the median non-urgent patient waited 17 minutes, the
median immediate patient 9. US EDs in these years saw most patients quickly.

In [ ]:
levels = ["1", "2", "3", "4", "5"]                     # the five triage levels; 0, 7 and 98 are special codes
triaged = F(triage).isin(levels)

wait_by_triage = by_level(wait, triage, levels)
display(wait_by_triage.style.format({c: MIN for c in ("mean", "q1", "median", "q3")}).hide(axis="index"))

anova = analysis([wait, triage], triaged, drop_missing=True).anova_oneway(group_by=triage, outcome=wait)
f, p, n, groups, tukey = anova_summary(anova)
print(f"One-way ANOVA of the wait across the five triage levels: F = {f:.1f}, p = {fmt_p(p)}, n = {n:,}")

ax = wait_by_triage.set_index("Triage level")[["median", "mean"]].plot(kind="bar", figsize=(6, 3.2), color=["#4C72B0", "#DD8452"], rot=0)
ax.set_ylabel("minutes"); ax.set_xlabel(""); ax.set_title("Wait time to provider, by triage level")
plt.tight_layout()
plt.show()

## Step 3. Length of visit and admission by triage level

With the same five levels: the median length of visit per level, and two cross-tabulations,
admission and arrival by ambulance by triage level (the `crosstab` helper: one `describe` per
level for the counts, the *Chi-squared Test* for the p-value).

*Expected:*

| Triage level | Median length of visit (min) | Admitted | Arrived by ambulance |
|---|---|---|---|
| 1 Immediate | 226 | 35.1% | 49.6% |
| 2 Emergent | 271 | 31.6% | 34.7% |
| 3 Urgent | 206 | 12.4% | 18.8% |
| 4 Semi-urgent | 114 | 1.6% | 6.6% |
| 5 Nonurgent | 98 | 4.9% | 8.2% |

Chi-square p < 0.001. Acuity drives both the stay and the outcome: a level-2 patient stays almost
three times longer than a level-4 patient and is twenty times more likely to be admitted. Note
the small level-5 group (2,398 visits) with a higher admission rate than level 4: triage is a
judgement, and non-urgent labels are sometimes wrong.

In [ ]:
lov_by_triage = by_level(lov, triage, levels)
adm = crosstab(triage, admitted, levels=levels)                 # % Yes = admitted
amb = crosstab(triage, ambulance, levels=levels, event="1")     # 1 = Yes in this CDC question item

table = pd.DataFrame({"median length of visit (min)": lov_by_triage.set_index("Triage level")["median"],
                      "% admitted": adm.set_index("Triage level")["% Yes"],
                      "% arrived by ambulance": amb.set_index("Triage level")["% Yes"]})
display(table.style.format(PCT))
print(f"Admission by triage level: chi2 = {adm.attrs['chi2']:.0f}, p = {fmt_p(adm.attrs['p'])}; "
      f"ambulance by triage level: chi2 = {amb.attrs['chi2']:.0f}, p = {fmt_p(amb.attrs['p'])}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
table["median length of visit (min)"].plot(kind="bar", ax=axes[0], color="#4C72B0", rot=0)
axes[0].set_ylabel("minutes"); axes[0].set_xlabel(""); axes[0].set_title("Median length of visit")
table[["% admitted", "% arrived by ambulance"]].plot(kind="bar", ax=axes[1], color=["#C44E52", "#8172B2"], rot=0)
axes[1].set_ylabel("% of visits"); axes[1].set_xlabel(""); axes[1].set_title("Admission and ambulance arrival")
plt.tight_layout()
plt.show()

## Step 4. Does the time of arrival matter?

All visits again. *One-way ANOVA* of the waiting time by arrival period (night, day, evening),
then an *Independent t-test* of the waiting time between weekend and weekday visits. The
platform's t-test pools the variances of the two groups.

*Expected:* medians 12 minutes at night, 14 in the day, 15 in the evening (ANOVA F about 17,
p < 0.001); weekend mean 31 versus weekday 36 minutes, medians 13 versus 15 (t about 10,
p < 0.001). Waits are shortest at night and on weekends, when fewer patients arrive. The
differences are statistically certain and operationally small.

In [ ]:
wait_by_period = by_level(wait, period)
display(wait_by_period.style.format({c: MIN for c in ("mean", "q1", "median", "q3")}).hide(axis="index"))
anova = analysis([wait, period], drop_missing=True).anova_oneway(group_by=period, outcome=wait)
f, p, n, *_ = anova_summary(anova)
print(f"One-way ANOVA of the wait across the three arrival periods: F = {f:.1f}, p = {fmt_p(p)}, n = {n:,}")

wait_by_weekend = by_level(wait, weekend)
display(wait_by_weekend.style.format({c: MIN for c in ("mean", "q1", "median", "q3")}).hide(axis="index"))
t = analysis([wait, weekend], drop_missing=True).t_test(variable=wait, group_by=weekend, group_a="1", group_b="0")
s = ttest_summary(t)
print(f"Independent t-test, weekend minus weekday: difference {s['mean_diff']:.1f} minutes "
      f"(95% CI {s['ci_lower']:.1f} to {s['ci_upper']:.1f}), t = {s['t_stat']:.1f}, p = {fmt_p(s['p'])}")

## Step 5. Admitted patients stay longer, and some are boarded

An *Independent t-test* of the length of visit between admitted and discharged visits, the
medians per group, then a `describe()` of the boarding time among admitted visits. Two more
`describe()` calls with a filter on the boarding time count the visits boarded for more than two
and more than four hours.

*Expected:* length of visit median 300 minutes when admitted versus 163 when not (means 482
versus 239, t about 59, p < 0.001). Boarding time is known for 7,970 admitted visits: median 36
minutes, quartiles 0 to 136, mean 171. About 27% of admitted patients waited more than two hours
for a bed after the admission decision, and 15% more than four hours.

In [ ]:
t = analysis([lov, admitted], drop_missing=True).t_test(variable=lov, group_by=admitted, group_a="1", group_b="0")
s = ttest_summary(t)
lov_by_admitted = by_level(lov, admitted)
display(lov_by_admitted.style.format({c: MIN for c in ("mean", "q1", "median", "q3")}).hide(axis="index"))
print(f"Independent t-test, admitted minus not admitted: difference {s['mean_diff']:.0f} minutes, "
      f"t = {s['t_stat']:.1f}, p = {fmt_p(s['p'])}")

inpatients = F(admitted) == "1"
board = numeric_summary(analysis([boarded], inpatients).describe(), [boarded])
display(board.style.format({c: MIN for c in ("mean", "sd", "q1", "median", "q3", "min", "max")}).hide(axis="index"))
n_boarded = int(board["n"].iloc[0])
over = {}
for hours in (2, 4):
    res = analysis([boarded], inpatients & (F(boarded) > 60 * hours)).describe()
    over[hours] = int(numeric_summary(res, [boarded])["n"].iloc[0])
print(f"Boarded more than 2 hours: {over[2]:,} of {n_boarded:,} admitted visits with a boarding time ({100 * over[2] / n_boarded:.1f}%); "
      f"more than 4 hours: {over[4]:,} ({100 * over[4] / n_boarded:.1f}%)")

## Step 6. Trends over the five years

One `describe()` per survey year for the length of visit and the waiting time, then the
cross-tabulation of leaving without being seen by year.

*Expected:* median length of visit rose from 165 minutes in 2018 to 191 in 2022 while the median
wait stayed around 14 to 16 minutes (11 in 2020, the pandemic year with fewer visits). Patients
leaving without being seen rose from 1.0% to 2.0% (chi-square 101, p < 0.001). These are signs of
growing ED crowding in the United States after the pandemic, visible even in an unweighted sample.

In [ ]:
trend = pd.DataFrame({"median length of visit": by_level(lov, year).set_index("Survey year")["median"],
                      "median wait": by_level(wait, year).set_index("Survey year")["median"]})
display(trend.style.format(MIN))
left = crosstab(year, lwbs)
show_crosstab(left, "Left without being seen, by survey year")

fig, ax = plt.subplots(figsize=(6, 3.2))
trend.plot(ax=ax, marker="o", color=["#4C72B0", "#DD8452"])
ax.set_ylim(0); ax.set_ylabel("minutes"); ax.set_xlabel(""); ax.set_title("Median visit length and wait, 2018 to 2022")
plt.tight_layout()
plt.show()

## Step 7. State the result

Write two or three sentences. A model answer:

> Among 86,864 sampled US ED visits from 2018 to 2022, the median wait to a provider was 14 minutes
> and the median visit lasted 175 minutes; both distributions are strongly skewed, so medians are
> the right summary. Waiting time fell with acuity (medians 9 to 17 minutes from triage level 1 to
> 5) while length of visit and admission rose with it (32 to 35% of level 1 and 2 patients admitted
> versus 1.6% of level 4). Admitted patients stayed a median of 5 hours in the ED, and a quarter of
> them boarded more than two hours after the admission decision. Between 2018 and 2022 the median
> visit lengthened by 26 minutes and the share of patients leaving without being seen doubled.
> These are unweighted sample figures; national estimates require the survey weights.

**Questions for discussion.** The mean waiting time is 35 minutes and the median 14: which one
would a hospital director want to see, and which one would a patient want to know? The ANOVA of
Step 2 has F about 20 and p < 0.001 for medians that differ by 8 minutes: what does a very small
p-value tell you here, and what does it not? Why might a level-5 "non-urgent" patient be admitted
more often than a level-4 patient?

**Optional extension.** Waiting time by payer: *One-way ANOVA* of the wait by Primary expected
payer after filtering out the unknown code (98), then look at the age distribution of each payer
group before interpreting any difference.

In [ ]:
payer = dm.variables["Primary expected payer"]
known_payer = F(payer).isin(["1", "2", "3", "4", "5", "6", "7"])
wait_by_payer = by_level(wait, payer, ["1", "2", "3", "4", "5", "6", "7"])
display(wait_by_payer.style.format({c: MIN for c in ("mean", "q1", "median", "q3")}).hide(axis="index"))
anova = analysis([wait, payer], known_payer, drop_missing=True).anova_oneway(group_by=payer, outcome=wait)
f, p, n, *_ = anova_summary(anova)
print(f"One-way ANOVA of the wait across the seven payer groups: F = {f:.1f}, p = {fmt_p(p)}, n = {n:,}")